In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# Google ADK

Chapter 10 is the Model Context Protocol: a client/server standard for exposing tools to any LLM. The book's hands-on section (pp. 153-159) has two examples, and both run through `adk web`, a browser UI, rather than a notebook:

1. **A filesystem agent.** The agent has no tool code of its own. Its `McpToolset` spawns the community `@modelcontextprotocol/server-filesystem` with `npx`, talks to it over stdin/stdout, and discovers its tools at runtime.
2. **A FastMCP greeter.** A 15-line Python MCP server exposes one `greet` tool over HTTP; a second agent consumes it as an MCP client.

The book's files live in `adk_agent_samples/` at the repository root, in the layout `adk web` expects:

```
adk_agent_samples/
├── mcp_agent/                  example 1
│   ├── __init__.py
│   ├── agent.py
│   └── mcp_managed_files/      the only folder the agent may touch
├── fastmcp_client_agent/       example 2, the client
│   ├── __init__.py
│   └── agent.py
└── fastmcp_server.py           example 2, the server
```

Each example below is given twice: the `adk web` walkthrough as the book intends it, then the same agent run inside this notebook with `InMemoryRunner` so the run is recorded here.

**Ported from the book.** The listings use the pre-1.0 MCP API:

| The book | google-adk 2.7.1 |
| --- | --- |
| `MCPToolset` | `McpToolset` (the old name still works but warns) |
| `from ...mcp_toolset import StdioServerParameters` | it is the `mcp` SDK's class; ADK wraps it: `StdioConnectionParams(server_params=StdioServerParameters(...), timeout=60)`. The default 5 s timeout is too short for `npx` to download the server on first use |
| `HttpServerParameters(url="http://localhost:8000")` | does not exist. `StreamableHTTPConnectionParams(url="http://127.0.0.1:8000/mcp")` — note the `/mcp` path FastMCP serves on |
| `model='gemini-2.0-flash'` | retired; `gemini-flash-latest` |
| `adk web` and the FastMCP server both on port 8000 | they cannot share it; run `adk web --port 8080` |

## MCP filesystem server over stdio

**Running it the book's way, with `adk web`.** One terminal is enough: the MCP server is a subprocess the agent starts itself.

1. Read `adk_agent_samples/mcp_agent/agent.py`. Three things to notice: the agent has no tools of its own; `npx` launches the filesystem server and the two talk over stdio; `TARGET_FOLDER_PATH` is passed to the server as its sandbox.
2. Start the UI:
   ```bash
   cd adk_agent_samples
   uv run adk web
   ```
   The last line should be `Uvicorn running on http://127.0.0.1:8000`. `.env` is found automatically — `adk web` walks up from the agent folder to the repository root.
3. Open http://127.0.0.1:8000 and pick **`mcp_agent`** — apps are listed by directory name, not by the agent's `name`.
4. Send `Show me the contents of this folder.` Expect 5-10 s of silence the first time while `npx` downloads the server. The terminal prints `Secure MCP Filesystem Server running on stdio`, then the reply lists `sample.txt`.
5. Open the **Events** tab on the left: the turn is a `function_call` (the model chose a tool it had never seen at import time), a `function_response` (the server's raw reply), then the text.
6. `Read the 'sample.txt' file.` — tool `read_text_file`. Then `Create a file called notes.md containing a three-item todo list for learning MCP.` — tool `write_file`. Check the folder from a second terminal: the file is really there.
7. `List the files in /Users/<you>` — the server refuses. The boundary is enforced by the server, not by the prompt.

The cell below is the same agent, driven by `InMemoryRunner` with the prompts from steps 4, 6 and 7.

In [3]:
# Example 1 (book p. 154): an agent whose only tools come from an MCP server.
# Prerequisite: Node.js. `npx` fetches @modelcontextprotocol/server-filesystem on first use.
import os
import sys
import warnings

# ADK 2.7.1 warns once per process about three experimental-but-default-on features on
# the MCP path, the first of them at import time. Two cannot be disabled (the decorator
# raises if they are), so the notices are filtered rather than switched off. The two env
# vars are real switches: the tool-declaration feature (see chapter 8) and Google mTLS,
# which ADK tries on every HTTP MCP connection unless told not to - pointless locally.
warnings.filterwarnings(
    "ignore",
    message=r"\[EXPERIMENTAL\] feature FeatureName\.(PLUGGABLE_AUTH|BASE_AUTHENTICATED_TOOL|_MCP_GRACEFUL_ERROR_HANDLING)",
)
os.environ.setdefault("ADK_DISABLE_JSON_SCHEMA_FOR_FUNC_DECL", "1")
os.environ.setdefault("GOOGLE_API_USE_CLIENT_CERTIFICATE", "false")

from google.adk.agents import LlmAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset, StdioConnectionParams
from google.genai.types import (
    AutomaticFunctionCallingConfig, Content, GenerateContentConfig, Part,
)
from mcp import StdioServerParameters

GEMINI_MODEL = "gemini-flash-latest"  # the book's gemini-2.0-flash is retired
NO_AFC = GenerateContentConfig(automatic_function_calling=AutomaticFunctionCallingConfig(disable=True))

# The same folder agent.py uses: the only directory the MCP server is allowed to touch.
TARGET_FOLDER_PATH = os.path.abspath("../adk_agent_samples/mcp_agent/mcp_managed_files")
os.makedirs(TARGET_FOLDER_PATH, exist_ok=True)


def filesystem_toolset() -> McpToolset:
    """The MCP filesystem server, started by the toolset as an npx subprocess."""
    return McpToolset(
        connection_params=StdioConnectionParams(
            server_params=StdioServerParameters(
                command="npx",
                args=["-y", "@modelcontextprotocol/server-filesystem", TARGET_FOLDER_PATH],
            ),
            timeout=60,  # first use downloads the server; the default 5 s is too short
        ),
        # The server's startup banner goes to the kernel's own stderr (the terminal that
        # launched Jupyter), not to this notebook.
    )


filesystem_agent = LlmAgent(
    model=GEMINI_MODEL,
    name="filesystem_assistant_agent",
    instruction=(
        "Help the user manage their files. You can list files, read files, and write files. "
        f"You are operating in the following directory: {TARGET_FOLDER_PATH}"
    ),
    tools=[filesystem_toolset()],
    generate_content_config=NO_AFC,
)


async def run_agent(agent, prompts: list[str]):
    """Runs each prompt in one session and prints every tool call and final reply."""
    runner = InMemoryRunner(agent, app_name="mcp_demo")
    session = await runner.session_service.create_session(app_name="mcp_demo", user_id="user")
    for prompt in prompts:
        print(f"\n>>> {prompt}")
        async for event in runner.run_async(
            user_id="user", session_id=session.id,
            new_message=Content(role="user", parts=[Part(text=prompt)]),
        ):
            for part in (event.content.parts if event.content and event.content.parts else []):
                if part.function_call:
                    print(f"  [tool call] {part.function_call.name}({dict(part.function_call.args)})")
            if event.is_final_response() and event.content and event.content.parts:
                print("  [agent]", "".join(p.text or "" for p in event.content.parts).strip())
    await agent.tools[0].close()  # stops the npx subprocess


# Tool discovery happens at runtime: nothing in this notebook defines these.
tools = await filesystem_agent.tools[0].get_tools()
print("Tools discovered from the MCP server:", [t.name for t in tools])

await run_agent(filesystem_agent, [
    "Show me the contents of this folder.",
    "Read the 'sample.txt' file.",
    "Create a file called notes.md containing a three-item todo list for learning MCP.",
    f"List the files in {os.path.expanduser('~')}",
])
print("\nOn disk now:", sorted(os.listdir(TARGET_FOLDER_PATH)))

Tools discovered from the MCP server: ['create_directory', 'directory_tree', 'edit_file', 'get_file_info', 'list_allowed_directories', 'list_directory', 'list_directory_with_sizes', 'move_file', 'read_file', 'read_media_file', 'read_multiple_files', 'read_text_file', 'search_files', 'write_file']

>>> Show me the contents of this folder.


  [tool call] list_directory_with_sizes({'path': '/Users/hodung/Workspace/ta/agentic-design-patterns/adk_agent_samples/mcp_agent/mcp_managed_files'})


  [agent] Here are the contents of the folder:

- **`notes.md`** (268 B)
- **`sample.txt`** (100 B)

Total: 2 files, 0 directories (368 B)

>>> Read the 'sample.txt' file.


  [tool call] read_text_file({'path': '/Users/hodung/Workspace/ta/agentic-design-patterns/adk_agent_samples/mcp_agent/mcp_managed_files/sample.txt'})


  [agent] Here are the contents of **`sample.txt`**:

```
Hello from sample.txt. This file lives in the folder the MCP filesystem server is allowed to touch.
```

>>> Create a file called notes.md containing a three-item todo list for learning MCP.


  [tool call] write_file({'path': '/Users/hodung/Workspace/ta/agentic-design-patterns/adk_agent_samples/mcp_agent/mcp_managed_files/notes.md', 'content': '# MCP Learning Todo List\n\n- [ ] 1. Read the Model Context Protocol (MCP) specification and core concepts\n- [ ] 2. Build and run a basic MCP server using a filesystem or tool integration\n- [ ] 3. Connect an LLM client/agent to the MCP server and test tool calls\n'})


  [agent] I have created `notes.md` with the following content:

```markdown
# MCP Learning Todo List

- [ ] 1. Read the Model Context Protocol (MCP) specification and core concepts
- [ ] 2. Build and run a basic MCP server using a filesystem or tool integration
- [ ] 3. Connect an LLM client/agent to the MCP server and test tool calls
```

>>> List the files in /Users/hodung


  [tool call] list_directory({'path': '/Users/hodung'})


  [agent] I cannot access `/Users/hodung`. 

Access is restricted strictly to the allowed directory:
`/Users/hodung/Workspace/ta/agentic-design-patterns/adk_agent_samples/mcp_agent/mcp_managed_files`

I can only view and manage files and subdirectories within that path.

On disk now: ['notes.md', 'sample.txt']


## Google ADK + Ollama

In [4]:
# Same filesystem agent, running on a local model served by Ollama through LiteLLM.
# Prerequisites:
#   ollama pull qwen2.5:7b
# Reuses filesystem_toolset and run_agent from the cell above. The MCP server does not
# care which model calls it: the tools are the same fourteen, discovered the same way.
from google.adk.models.lite_llm import LiteLlm

os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")
# 'ollama_chat', not 'ollama': it is the provider that handles tool calling correctly.
OLLAMA_MODEL = "ollama_chat/qwen2.5:7b"

ollama_filesystem_agent = LlmAgent(
    model=LiteLlm(model=OLLAMA_MODEL),
    name="filesystem_assistant_agent",
    instruction=(
        "Help the user manage their files. You can list files, read files, and write files. "
        f"You are operating in the following directory: {TARGET_FOLDER_PATH}"
    ),
    tools=[filesystem_toolset()],
)

await run_agent(ollama_filesystem_agent, [
    "Show me the contents of this folder.",
    "Read the 'sample.txt' file.",
])


>>> Show me the contents of this folder.


  [tool call] list_directory({'path': '/Users/hodung/Workspace/ta/agentic-design-patterns/adk_agent_samples/mcp_agent/mcp_managed_files'})


  [agent] The folder contains the following files:
- notes.md
- sample.txt

>>> Read the 'sample.txt' file.


  [tool call] read_text_file({'path': '/Users/hodung/Workspace/ta/agentic-design-patterns/adk_agent_samples/mcp_agent/mcp_managed_files/sample.txt'})


  [agent] The content of the 'sample.txt' file is:
```
Hello from sample.txt. This file lives in the folder the MCP filesystem server is allowed to touch.
```


## FastMCP server over HTTP

Example 2 turns the arrangement around: the MCP server is a separate process you write and start yourself, and the agent reaches it over HTTP. The whole server is `adk_agent_samples/fastmcp_server.py`:

```python
from fastmcp import FastMCP

mcp_server = FastMCP()

@mcp_server.tool
def greet(name: str) -> str:
    """
    Generates a personalized greeting.

    Args:
        name: The name of the person to greet.

    Returns:
        A greeting string.
    """
    return f"Hello, {name}! Nice to meet you."

if __name__ == "__main__":
    mcp_server.run(transport="http", host="127.0.0.1", port=8000)
```

The docstring and the `name: str` hint *are* the tool's schema; FastMCP generates it, and nothing else describes the tool to the model.

**Running it the book's way, with `adk web`.** Two terminals, two ports.

1. Terminal 1, the server:
   ```bash
   uv run python adk_agent_samples/fastmcp_server.py
   ```
   Expect `Starting MCP server ... with transport 'http' on http://127.0.0.1:8000/mcp`. Leave it running; it logs every request the agent makes.
2. Terminal 2, the UI on a **different port** — `adk web` also defaults to 8000, and the book runs both there. If `adk web` gets 8000 first, the agent's `POST /mcp` lands on `adk web` itself and comes back `404 Not Found`, the toolset reports `Failed to get tools`, and the model answers without any tools.
   ```bash
   cd adk_agent_samples
   uv run adk web --port 8080
   ```
3. Open http://127.0.0.1:8080, pick **`fastmcp_client_agent`**, send `Greet John Doe.` The reply is the server's string verbatim, and terminal 1 shows `"POST /mcp HTTP/1.1" 200 OK` — the tool call leaving the agent process.
4. Stop the server with Ctrl+C in terminal 1 and send `Greet Jane.` The agent has no tools now. Restart the server and it works again. The capability lives in the server, not the agent.

The cell below starts the same server as a subprocess, runs the same agent against it, and stops it — so the notebook is self-contained.

In [5]:
# Example 2 (book pp. 157-159): an agent consuming a FastMCP server as an MCP client.
import contextlib
import socket
import subprocess
import time

from google.adk.tools.mcp_tool.mcp_toolset import StreamableHTTPConnectionParams

FASTMCP_SERVER_URL = "http://127.0.0.1:8000/mcp"  # FastMCP serves Streamable HTTP at /mcp


@contextlib.contextmanager
def fastmcp_server():
    """Runs adk_agent_samples/fastmcp_server.py for the duration of the block."""
    proc = subprocess.Popen(
        [sys.executable, "../adk_agent_samples/fastmcp_server.py"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    try:
        for _ in range(50):  # wait until the port answers
            with socket.socket() as s:
                if s.connect_ex(("127.0.0.1", 8000)) == 0:
                    break
            time.sleep(0.2)
        print("FastMCP server up on", FASTMCP_SERVER_URL)
        yield
    finally:
        proc.terminate()
        proc.wait()
        print("FastMCP server stopped.")


def greet_toolset() -> McpToolset:
    """The FastMCP server reached over HTTP, exposing only its greet tool."""
    return McpToolset(
        connection_params=StreamableHTTPConnectionParams(url=FASTMCP_SERVER_URL),
        tool_filter=["greet"],
    )


greeter_agent = LlmAgent(
    model=GEMINI_MODEL,
    name="fastmcp_greeter_agent",
    instruction='You are a friendly assistant that can greet people by their name. Use the "greet" tool.',
    tools=[greet_toolset()],
    generate_content_config=NO_AFC,
)

with fastmcp_server():
    tools = await greeter_agent.tools[0].get_tools()
    print("Tools discovered from the MCP server:", [t.name for t in tools])
    await run_agent(greeter_agent, ["Greet John Doe."])

FastMCP server up on http://127.0.0.1:8000/mcp
Tools discovered from the MCP server: ['greet']

>>> Greet John Doe.


  [tool call] greet({'name': 'John Doe'})


  [agent] Hello, John Doe! Nice to meet you.
FastMCP server stopped.


## Google ADK + Ollama

In [6]:
# Same greeter, running on a local model served by Ollama through LiteLLM.
# Reuses fastmcp_server, greet_toolset and run_agent from the cells above.
ollama_greeter_agent = LlmAgent(
    model=LiteLlm(model=OLLAMA_MODEL),
    name="fastmcp_greeter_agent",
    instruction='You are a friendly assistant that can greet people by their name. Use the "greet" tool.',
    tools=[greet_toolset()],
)

with fastmcp_server():
    await run_agent(ollama_greeter_agent, ["Greet John Doe."])

FastMCP server up on http://127.0.0.1:8000/mcp

>>> Greet John Doe.


  [tool call] greet({'name': 'John Doe'})


  [agent] Hello, John Doe! Nice to meet you.
FastMCP server stopped.
